# Appendix D: Common Equations

Printed source span verified before authoring: Appendix D is listed as printed pages 603-608. The PDF text layer contains printed pages 603-607 on PDF pages 618-622; printed page 608 appears to be an omitted blank page before the bibliography. This notebook is an original curated formula sheet. It keeps the spirit of the appendix by collecting frequently used relationships, but it pairs each formula group with an executable check.

The goal is not to memorize a wall of equations. The goal is to know which invariant each equation should preserve. A projection should recompose with its rejection. A rotor should preserve norm and send the chosen source vector to the chosen target vector. A conformal point embedding should be null and should recover Euclidean squared distance through the conformal inner product. Formula sheets are most useful when every line carries a test idea.


## Translation Guide

Read every formula in this notebook in three columns: notation, implementation, and invariant. Notation gives the compact mathematical expression. Implementation says which local helper or product realizes it. The invariant says what must be true if the formula was applied under the right assumptions.

The assumptions matter. Duality needs a nondegenerate pseudoscalar. Blade inversion needs a non-null blade. The shortest rotor formula assumes unit input vectors and excludes the exactly opposite case where the shortest plane is not unique. Projection formulas assume the target blade has an inverse. Conformal distance formulas assume normalized conformal points under the model's inner product.

Appendix D in the book is broad; this notebook is selective. It focuses on equations that can be checked in a few lines and that are repeatedly useful across the course notebooks.


## Route

We begin by saving the verified source audit and a compact formula catalog. Then we compute the involution sign table, which is a frequent source of sign bugs. Next, we run product definition checks, duality and inverse checks, projection and rotor checks, exponential special cases, and conformal point checks. The final sanity cell asserts that all reference artifacts exist and that the key invariants survived execution.


In [ ]:
from pathlib import Path
import json, math, sys
import numpy as np
import pandas as pd
PROJECT_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "utils" / "ga").exists(): PROJECT_ROOT = candidate; break
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from utils.ga import Algebra, conformal_distance_squared, conformal_inner, conformal_point
from utils.appendix_reference import (
    basis_blade_records, formula_catalog, grade_selection_records, projection_onto_blade,
    rejection_from_blade, shortest_rotor_between_unit_vectors, sign_change_table,
    source_span_records, write_csv_artifact, write_json_artifact,
)
pd.set_option("display.max_columns", 14)
APPENDIX_KEY = "appendix-d"
artifact_paths = {}
def remember(label, path): artifact_paths[label] = str(Path(path)); return Path(path)
print("project", PROJECT_ROOT)


In [ ]:
source_rows = source_span_records()
source_payload = {"pdf":"Geometric Algebra for Computer Science.pdf","verification_tools":["pdfinfo","pdftotext -layout"],"pdf_page_count":640,"note":"Printed pages are authoritative; omitted blank pages are listed explicitly.","records":source_rows}
remember("source_span_json", write_json_artifact(["source"], "source-span-audit.json", source_payload))
remember("source_span_csv", write_csv_artifact(["source"], "source-span-audit.csv", source_rows))
pd.DataFrame(source_rows)


## Curated Formula Catalog

A useful formula catalog is short enough to scan and specific enough to test. The table below is not a transcription of the appendix tables. It is a curated working set for the notebooks: grade selection rules, common operators, involution signs, a rotor construction, and a conformal point identity. Each row includes the check that will make the formula meaningful in code.


In [ ]:
catalog_rows = formula_catalog()
remember("formula_catalog_csv", write_csv_artifact([APPENDIX_KEY,"tables"], "formula-catalog.csv", catalog_rows))
remember("formula_catalog_json", write_json_artifact([APPENDIX_KEY,"tables"], "formula-catalog.json", catalog_rows))
pd.DataFrame(catalog_rows)


## Involution Signs

Reversion, grade involution, and Clifford conjugation are simple sign patterns, but simple sign patterns are exactly the things that get copied incorrectly. The table below gives the multiplier by grade. It can be used as a quick reference when reading rotor inverses, duality formulas, and sandwich operations.

The executable check compares the table with actual basis blades in a four-dimensional Euclidean algebra. Bivectors and trivectors reverse sign; odd grades change sign under grade involution; Clifford conjugation combines both patterns.


In [ ]:
sign_rows = sign_change_table(8)
remember("sign_table_csv", write_csv_artifact([APPENDIX_KEY,"tables"], "involution-signs.csv", sign_rows))
alg4 = Algebra([1,1,1,1], names=["e1","e2","e3","e4"])
blade_records = basis_blade_records(alg4)
remember("basis_blade_signs_csv", write_csv_artifact([APPENDIX_KEY,"tables"], "basis-blade-signs.csv", blade_records))
pd.DataFrame(sign_rows)


## Product Definitions By Grade Selection

The most reusable equations in the appendix are the product definitions by grade selection. This cell uses one vector and one bivector, computes the geometric product, and retrieves outer product, left contraction, right contraction, and scalar product from it. The same table is also a compact reminder of when a requested product is zero because the target grade is impossible.


In [ ]:
alg = Algebra([1, 1, 1], names=["e1", "e2", "e3"])
e1, e2, e3 = alg.basis()
a = e1 + 2 * e3
B = e1.wedge(e2)
product_rows = grade_selection_records("a", a, "B", B)
assert all(row["matches_direct"] for row in product_rows)
remember("product_definitions_csv", write_csv_artifact([APPENDIX_KEY,"tables"], "product-definitions.csv", product_rows))
pd.DataFrame(product_rows)


## Inverse, Dual, And Undual

Inverses and duals are convenient only when their assumptions hold. In a nondegenerate Euclidean algebra the pseudoscalar is invertible, so the dual of a plane blade is the perpendicular vector up to orientation. Applying undual should recover the original blade. For a non-null blade, multiplying by its inverse should recover the scalar one.

These are small checks, but they protect a lot of formulas. Projection, meet, join, cross product, and many model-specific identities quietly depend on these operations.


In [ ]:
I = alg.pseudoscalar(); e12 = e1.wedge(e2)
dual_e12 = e12.dual(); undual_e12 = dual_e12.undual(); inverse_e12 = e12.inverse_blade()
inverse_payload = {"I":repr(I),"e12":repr(e12),"dual_e12":repr(dual_e12),"undual_e12":repr(undual_e12),"undual_recovers_up_to_sign":undual_e12.almost_equal(e12) or undual_e12.almost_equal(-e12),"e12_inverse":repr(inverse_e12),"e12_times_inverse":repr(e12.gp(inverse_e12))}
assert e12.gp(inverse_e12).almost_equal(alg.scalar(1))
assert inverse_payload["undual_recovers_up_to_sign"]
remember("inverse_dual_json", write_json_artifact([APPENDIX_KEY,"checks"], "inverse-dual-check.json", inverse_payload))
inverse_payload


## Projection, Reflection, And Rotors

Three common operator equations are worth keeping together. Projection onto a blade and rejection from a blade should add back to the original vector. Reflection in a unit normal should flip the normal component and preserve tangential components. The shortest rotor from one unit vector to another should be unit and should map the first vector to the second by sandwiching.

These checks cover different product styles: contractions for projection, geometric products for reflection, and even multivectors for rotors.


In [ ]:
x_vec = 2 * e1 + 3 * e2 + 4 * e3
plane = e1.wedge(e2)
proj = projection_onto_blade(x_vec, plane); rej = rejection_from_blade(x_vec, plane)
reflected = -e3.gp(x_vec).gp(e3)
target = (e1 + e2) / math.sqrt(2)
R = shortest_rotor_between_unit_vectors(e1, target)
rotated = R.gp(e1).gp(R.reverse())
operator_payload = {"projection":repr(proj),"rejection":repr(rej),"recomposes":(proj+rej).almost_equal(x_vec),"reflection_in_e3":repr(reflected),"rotor":repr(R),"rotor_unit_check":repr(R.gp(R.reverse())),"rotated_e1":repr(rotated),"rotor_hits_target":rotated.almost_equal(target)}
assert operator_payload["recomposes"] and operator_payload["rotor_hits_target"] and R.gp(R.reverse()).almost_equal(alg.scalar(1))
remember("operators_json", write_json_artifact([APPENDIX_KEY,"checks"], "projection-reflection-rotor.json", operator_payload))
operator_payload


## Exponential Special Cases

The appendix formula sheet includes exponential, sine, and cosine behavior for blades with negative, zero, and positive square. We can check the exponential cases with a short power series. A Euclidean unit bivector squares to minus one and produces circular functions. A null vector squares to zero and truncates the exponential after the linear term. A positive-square vector produces hyperbolic functions.

These checks are not a replacement for a robust exponential implementation, but they are valuable reference tests for the three qualitative cases.


In [ ]:
def exp_series(A, terms=18):
    result = A.algebra.scalar(1); power = A.algebra.scalar(1); factorial = 1
    for k in range(1, terms):
        power = power.gp(A); factorial *= k; result = result + power * (1 / factorial)
    return result
theta = 0.4
unit_biv = e1.wedge(e2)
circular = exp_series(unit_biv * theta)
circular_expected = alg.scalar(math.cos(theta)) + unit_biv * math.sin(theta)
null_alg = Algebra([-1, 1], names=["t", "x"])
t, x = null_alg.basis(); null_vec = (t + x) / math.sqrt(2)
nilpotent = exp_series(null_vec * 0.7); nilpotent_expected = null_alg.scalar(1) + null_vec * 0.7
hyper = exp_series(e1 * theta); hyper_expected = alg.scalar(math.cosh(theta)) + e1 * math.sinh(theta)
exponential_payload = {"circular_matches":circular.almost_equal(circular_expected, tol=1e-10),"nilpotent_matches":nilpotent.almost_equal(nilpotent_expected, tol=1e-10),"hyperbolic_matches":hyper.almost_equal(hyper_expected, tol=1e-10),"circular":repr(circular),"nilpotent":repr(nilpotent),"hyperbolic":repr(hyper)}
assert all(exponential_payload[key] for key in ["circular_matches","nilpotent_matches","hyperbolic_matches"])
remember("exponential_cases_json", write_json_artifact([APPENDIX_KEY,"checks"], "exponential-special-cases.json", exponential_payload))
exponential_payload


## Conformal Point Equations, Pitfalls, And Takeaways

The conformal point formula is one of the most useful common equations in the later chapters. A Euclidean vector `x` is embedded as a null point in a higher-dimensional space. The conformal inner product between two normalized points recovers Euclidean squared distance through `-2 P(x) dot P(y)`.

Formula sheets fail when assumptions are invisible. The compact equation may be right and still be wrong for the object at hand: the blade may be null, the metric may be degenerate, the rotor inputs may not be unit, or the conformal point may not be normalized. Keep a formula catalog, but keep invariant checks beside it. Prefer formulas that name their grade selection. When using inverses, assert invertibility. When using model equations, test the model's defining identities first. A common equation becomes a reliable tool only after it survives a small executable example.


In [ ]:
p = np.array([1.0, 2.0, -1.0]); q = np.array([-2.0, 0.5, 3.0])
P = conformal_point(p); Q = conformal_point(q)
conformal_payload = {"P_inner_P":conformal_inner(P,P),"Q_inner_Q":conformal_inner(Q,Q),"distance_squared_from_model":conformal_distance_squared(p,q),"distance_squared_direct":float(np.dot(p-q,p-q))}
assert abs(conformal_payload["P_inner_P"]) < 1e-12
assert abs(conformal_payload["Q_inner_Q"]) < 1e-12
assert abs(conformal_payload["distance_squared_from_model"] - conformal_payload["distance_squared_direct"]) < 1e-12
remember("conformal_common_json", write_json_artifact([APPENDIX_KEY,"checks"], "conformal-point-equations.json", conformal_payload))
conformal_payload


## Reference Note: Formula Sheet Discipline

The most useful formula sheets are boring in the best way: every identity names its preconditions and every precondition has a small test. Before applying a dual formula, check that the pseudoscalar is invertible. Before applying a projection formula, check that the target blade is not null. Before applying a rotor formula, check that the inputs are unit vectors and that the reverse really is the inverse. Before applying a conformal formula, check the null embedding and distance identity. This habit turns Appendix D from a list of remembered equations into a dependable engineering checklist. It also makes cross-text notation differences less dangerous, because each imported equation has to earn its place by preserving a visible invariant.


In [ ]:
assert all(row["matches_direct"] for row in product_rows)
assert operator_payload["rotor_hits_target"]
assert abs(conformal_payload["P_inner_P"]) < 1e-12
sanity_payload = {"appendix":"D", "artifact_count_before_sanity":len(artifact_paths), "artifacts":artifact_paths}
sanity_path = remember("final_sanity", write_json_artifact([APPENDIX_KEY,"checks"], "final-sanity.json", sanity_payload))
missing = [str(path) for path in map(Path, artifact_paths.values()) if not Path(path).exists()]
assert not missing, missing
print(json.dumps({"appendix":"D", "artifact_count":len(artifact_paths), "sanity":str(sanity_path)}, indent=2))
